# 08 Target mixture (D5)

Instead of betting on one intended target, score each pass toward all 10 teammates and weight by P(teammate is the target): P(complete) = sum_j q_j * P(complete | aimed at j). Code: `features.build_target_mixture_features`, `models.run_lomo_target_mixture`. This is now the headline model.

In [1]:
import pandas as pd
from passmodel.load import REPO_ROOT

RESULTS = REPO_ROOT / "results"
m = pd.read_csv(RESULTS / "metrics_per_fold.csv")

## Held-out log loss: single target vs target mixture

In [2]:
m[m["model"].isin(["xgboost (single target)", "xgboost (target mixture)"])].pivot(index="model", columns="fold", values="log_loss")

fold,1,2,3,mean
model,,,,
xgboost (single target),0.5201,0.4186,0.4541,0.4643
xgboost (target mixture),0.5037,0.4118,0.4504,0.4553


## Paired bootstrap: mixture minus single target (negative = mixture better)

In [3]:
ci = pd.read_csv(RESULTS / "target_mixture_ci.csv")
ci[ci["metric"] == "delta_log_loss"]

,model,fold,metric,estimate,ci_low,ci_high
3,xgboost (target mixture),1,delta_log_loss,-0.0164,-0.0304,-0.0027
7,xgboost (target mixture),2,delta_log_loss,-0.0068,-0.0172,0.0029
11,xgboost (target mixture),3,delta_log_loss,-0.0037,-0.0144,0.0063


## Did it fix the failure mode? The single-target model's 10 biggest surprises, under both models

In [4]:
pd.read_csv(RESULTS / "target_mixture_on_surprises.csv")

,passes,single_target_mean_p,target_mixture_mean_p,target_was_receiver
0,single-target model's 10 riskiest completed,0.158,0.207,2
1,single-target model's 10 safest failed,0.982,0.937,0


## How spread out are the target probabilities?
If the top teammate usually took nearly all the weight, the mixture would barely differ from the single-target model.

In [5]:
p = pd.read_csv(RESULTS / "predictions.csv")
w = p[p["model"].isin(["xgboost (single target)", "xgboost (target mixture)"])].pivot_table(index=["match_id", "event_id"], columns="model", values="p_complete")
(w["xgboost (target mixture)"] - w["xgboost (single target)"]).describe().round(3)

count    3344.000
mean       -0.004
std         0.075
min        -0.372
25%        -0.043
50%        -0.010
75%         0.028
max         0.393
dtype: float64